# Kubernetes 第3周：调度、管理与运维

> **学习目标**：理解调度机制，配置 HPA 自动伸缩，用 Helm 打包应用，掌握 RBAC

---

## 调度机制：Pod 最终落在哪个节点？

当你创建一个 Pod 时，K8s Scheduler 执行三步决策：

```
1. Filter（过滤）  →  筛选出能运行此 Pod 的节点
     - 节点资源够不够？（CPU/内存）
     - 端口冲突吗？
     - nodeSelector 匹配吗？
     - 污点(Taint)容忍吗？

2. Score（打分）   →  给候选节点打分
     - 资源剩余越多分数越高（LeastRequestedPriority）
     - 尽量分散同一应用的 Pod（PodAntiAffinity）
     - 尽量集中在同一区域（PodAffinity，减少网络延迟）

3. Bind（绑定）    →  选最高分节点，把 Pod 调度过去
```

### 控制调度的四种方式

| 方式 | 说明 | 强制程度 | 用法 |
|------|------|----------|------|
| **nodeSelector** | 简单的节点标签匹配 | 硬性 | `nodeSelector: {disk: ssd}` |
| **nodeAffinity** | 灵活的节点亲和性 | 硬性/软性 | 支持 In/NotIn/Exists 等操作符 |
| **podAffinity** | Pod 拉近（同一节点/区域） | 软性 | 减少延迟 |
| **podAntiAffinity** | Pod 推开（分散部署） | 软性 | 提高可用性 |
| **Taint/Toleration** | 节点"排斥" + Pod"容忍" | 硬性 | 专用节点 |

In [ ]:
! mkdir -p /tmp/k8s-demo

%%writefile /tmp/k8s-demo/affinity-demo.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web-affinity
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      affinity:
        podAntiAffinity:                # 反亲和：让副本分散
          preferredDuringSchedulingIgnoredDuringExecution:
          - weight: 100
            podAffinityTerm:
              labelSelector:
                matchLabels:
                  app: web
              topologyKey: "kubernetes.io/hostname"  # 按节点分散
      containers:
      - name: nginx
        image: nginx:alpine

print("配置了 podAntiAffinity：同一应用的 Pod 尽量不在同一节点")
print("这样即使一个节点挂了，其他副本还在其他节点上运行")

---

## 资源管理：requests 和 limits

如果不设资源限制，一个 Pod 就可能吃光整个节点的 CPU/内存，影响其他 Pod。

```yaml
resources:
  requests:      # 调度时"预留"的资源
    cpu: "250m"      # 0.25 核
    memory: "256Mi"  # 256 MB
  limits:        # 运行时"上限"
    cpu: "500m"      # 最多用 0.5 核
    memory: "512Mi"  # 最多用 512 MB
```

| 值 | 含义 | 超限后果 |
|------|------|----------|
| **requests** | 调度时保证的最小资源 | 保证有这么多 |
| **limits** | 运行时允许的最大资源 | CPU：被 throttle（性能下降）；内存：OOMKilled（容器被杀） |

### QoS 等级（由 requests/limits 自动决定）

| QoS | 条件 | 被杀优先级 |
|-----|------|-----------|
| **Guaranteed** | requests == limits（都设了且相等） | 最低 |
| **Burstable** | requests < limits（设了但不相等） | 中 |
| **BestEffort** | 没设 requests 和 limits | 最高（先被杀） |

In [ ]:
%%writefile /tmp/k8s-demo/deploy-with-resources.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-resourced
spec:
  replicas: 2
  selector:
    matchLabels:
      app: api
  template:
    metadata:
      labels:
        app: api
    spec:
      containers:
      - name: api
        image: python:3.12-slim
        command: ["python", "-c", "import time; [time.sleep(999) for _ in iter(int, 1)]"]
        resources:
          requests:
            cpu: "100m"
            memory: "128Mi"
          limits:
            cpu: "200m"
            memory: "256Mi"

! kubectl apply -f /tmp/k8s-demo/deploy-with-resources.yaml 2>/dev/null
! sleep 3
! kubectl get pods -l app=api -o wide 2>/dev/null
! kubectl describe pod -l app=api 2>/dev/null | grep -A5 "QoS"

In [ ]:
# 查看节点资源使用
! kubectl top pods 2>/dev/null || echo "metrics-server 未安装，运行: kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml"

# 清理
! kubectl delete deployment api-resourced --wait=false 2>/dev/null

---

## 健康检查：liveness / readiness / startup

K8s 不只是看"进程还在不在"，还能通过 HTTP/TCP/命令来判断"应用是否真正健康"。

| 探针 | 问题 | 失败后果 | 使用场景 |
|------|------|----------|----------|
| **livenessProbe** | "还活着吗？" | **重启容器** | 检测死锁、无响应 |
| **readinessProbe** | "能接流量吗？" | **从 Service 摘除** | 预热、依赖未就绪 |
| **startupProbe** | "启动完了吗？" | **阻塞 liveness/readiness** | 慢启动应用 |

**关键区别**：liveness 失败 → 杀容器重建；readiness 失败 → 不给它流量但保留容器。

In [ ]:
%%writefile /tmp/k8s-demo/deploy-healthcheck.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: healthy-app
spec:
  replicas: 2
  selector:
    matchLabels:
      app: healthy
  template:
    metadata:
      labels:
        app: healthy
    spec:
      containers:
      - name: app
        image: python:3.12-slim
        command: ["python", "-c"]
        args:
        - |
          from http.server import HTTPServer, BaseHTTPRequestHandler
          import time
          start_time = time.time()
          class H(BaseHTTPRequestHandler):
              def do_GET(self):
                  if self.path == "/health" or self.path == "/healthz":
                      self.send_response(200)
                      self.end_headers()
                      self.wfile.write(b"ok")
                  elif self.path == "/ready":
                      self.send_response(200)
                      self.end_headers()
                      self.wfile.write(b"ready")
                  else:
                      self.send_response(200)
                      self.end_headers()
                      self.wfile.write(f"uptime: {time.time()-start_time:.0f}s".encode())
          HTTPServer(("0.0.0.0", 8000), H).serve_forever()
        ports:
        - containerPort: 8000
        # 存活探针：每隔 10 秒检查 /health，失败 3 次就重启
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 5    # 启动后等 5 秒才开始检查
          periodSeconds: 10          # 每 10 秒检查一次
          timeoutSeconds: 3          # 超时 3 秒
          failureThreshold: 3        # 连续失败 3 次算失败
        # 就绪探针：检查 /ready，未就绪时不分配流量
        readinessProbe:
          httpGet:
            path: /ready
            port: 8000
          initialDelaySeconds: 3
          periodSeconds: 5

print("健康检查配置：")
print("  liveness: GET /health，失败 3 次重启容器")
print("  readiness: GET /ready，失败则从 Service 摘除")

---

## HPA：水平自动伸缩

HPA（Horizontal Pod Autoscaler）根据 CPU/内存/自定义指标自动调整 Pod 副本数。

```yaml
HPA:
  minReplicas: 2     # 最少 2 个
  maxReplicas: 10    # 最多 10 个
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        averageUtilization: 70   # 平均 CPU 超过 70% 就扩容
```

HPA 的计算公式：
```
期望副本数 = ceil(当前副本数 × (当前指标值 / 目标指标值))

# 例如：当前 2 个副本，平均 CPU 90%，目标是 70%
期望副本数 = ceil(2 × (90 / 70)) = ceil(2.57) = 3
```

In [ ]:
%%writefile /tmp/k8s-demo/hpa-demo.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hpa-app
spec:
  replicas: 1
  selector:
    matchLabels:
      app: hpa-app
  template:
    metadata:
      labels:
        app: hpa-app
    spec:
      containers:
      - name: app
        image: python:3.12-slim
        command: ["python", "-c"]
        args:
        - |
          # 模拟 CPU 负载的应用
          from http.server import HTTPServer, BaseHTTPRequestHandler
          class H(BaseHTTPRequestHandler):
              def do_GET(self):
                  if self.path == "/load":
                      # 计算密集型操作，制造 CPU 负载
                      total = sum(i*i for i in range(5000000))
                      self.send_response(200)
                      self.end_headers()
                      self.wfile.write(f"计算完成: {total}\n".encode())
                  else:
                      self.send_response(200)
                      self.end_headers()
                      self.wfile.write(b"ok\n")
          HTTPServer(("0.0.0.0", 8000), H).serve_forever()
        resources:
          requests:
            cpu: "100m"
          limits:
            cpu: "500m"
        ports:
        - containerPort: 8000

! kubectl apply -f /tmp/k8s-demo/hpa-demo.yaml 2>/dev/null

# 创建 HPA
! kubectl autoscale deployment hpa-app --cpu-percent=50 --min=1 --max=5 2>/dev/null
! kubectl get hpa 2>/dev/null

In [ ]:
# 如果没有 metrics-server，HPA 会显示 <unknown>
# 安装 metrics-server（kind 环境）：
print("安装 metrics-server：")
print("  kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml")
print("  # 然后等待 metrics 可用：")
print("  kubectl get --raw '/apis/metrics.k8s.io/v1beta1/nodes'")

# 清理
! kubectl delete hpa hpa-app --wait=false 2>/dev/null
! kubectl delete deployment hpa-app --wait=false 2>/dev/null

---

## Helm：K8s 的包管理器

你手动写了一堆 YAML（Deployment、Service、ConfigMap、Secret...），换个环境要改一堆值。

**Helm 把相关的 K8s 资源打包成一个 Chart（包），用模板变量和 values 文件管理差异化配置。**

```
Chart 结构：
myapp/
├── Chart.yaml          # 包的元数据（名称、版本）
├── values.yaml         # 默认配置值
├── templates/          # 模板文件
│   ├── deployment.yaml
│   ├── service.yaml
│   ├── configmap.yaml
│   └── ingress.yaml
└── charts/             # 依赖的子 Chart
```

模板语法示例：

```yaml
# templates/deployment.yaml
replicas: {{ .Values.replicaCount }}
image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
```

```yaml
# values.yaml
replicaCount: 3
image:
  repository: myapp
  tag: "1.0"
```

In [ ]:
# 安装 Helm（如果还没装）
# curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash
! helm version 2>/dev/null || echo "Helm 未安装"

In [ ]:
# 创建一个简单的 Chart
! mkdir -p /tmp/k8s-demo/helm-demo/templates

%%writefile /tmp/k8s-demo/helm-demo/Chart.yaml
apiVersion: v2
name: demo-app
description: 一个演示用的 Helm Chart
version: 0.1.0
appVersion: "1.0"

%%writefile /tmp/k8s-demo/helm-demo/values.yaml
replicaCount: 2
image:
  repository: nginx
  tag: alpine
service:
  port: 80
  type: ClusterIP
ingress:
  enabled: false
  host: demo.local

%%writefile /tmp/k8s-demo/helm-demo/templates/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}
    spec:
      containers:
      - name: app
        image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
        ports:
        - containerPort: {{ .Values.service.port }}

%%writefile /tmp/k8s-demo/helm-demo/templates/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}-svc
spec:
  type: {{ .Values.service.type }}
  selector:
    app: {{ .Release.Name }}
  ports:
  - port: {{ .Values.service.port }}
    targetPort: {{ .Values.service.port }}

print("Helm Chart 已创建")

In [ ]:
# 安装 Chart
! helm install my-release /tmp/k8s-demo/helm-demo 2>/dev/null
! sleep 3
! helm list 2>/dev/null
! kubectl get deploy,svc 2>/dev/null

In [ ]:
# 覆盖默认值
! helm upgrade my-release /tmp/k8s-demo/helm-demo \
  --set replicaCount=4 \
  --set image.tag=latest 2>/dev/null
! kubectl get deploy my-release -o jsonpath="{.spec.replicas}" 2>/dev/null
print("  ← 副本数已变为 4")

# 回滚
! helm rollback my-release 1 2>/dev/null
! sleep 2
! kubectl get deploy my-release -o jsonpath="{.spec.replicas}" 2>/dev/null
print("  ← 回滚到第 1 版，副本数恢复为 2")

# 卸载
! helm uninstall my-release 2>/dev/null

---

## RBAC：权限控制

K8s 的权限模型基于三个概念：

```
Who（谁）         →  ServiceAccount（人或机器的身份）
Can do What（能干什么） →  Role / ClusterRole（权限集合）
On What（对什么） →  RoleBinding / ClusterRoleBinding（把身份和权限绑在一起）
```

```
ServiceAccount ── RoleBinding ── Role ── 可以 get/list Pod
   (身份)          (绑定)       (权限)
```

In [ ]:
%%writefile /tmp/k8s-demo/rbac-demo.yaml
# 1. 创建 ServiceAccount
apiVersion: v1
kind: ServiceAccount
metadata:
  name: readonly-user
---
# 2. 创建 Role：只能读 Pod
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
rules:
- apiGroups: [""]           # "" 表示核心 API 组
  resources: ["pods"]
  verbs: ["get", "list", "watch"]    # 只读
---
# 3. 绑定：把 Role 赋给 ServiceAccount
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: read-pods-binding
subjects:
- kind: ServiceAccount
  name: readonly-user
roleRef:
  kind: Role
  name: pod-reader
  apiGroup: rbac.authorization.k8s.io

! kubectl apply -f /tmp/k8s-demo/rbac-demo.yaml 2>/dev/null
print("RBAC 已配置：readonly-user 只能读取 Pod")

# 验证权限
! kubectl auth can-i get pods --as=system:serviceaccount:default:readonly-user 2>/dev/null
! kubectl auth can-i delete pods --as=system:serviceaccount:default:readonly-user 2>/dev/null

In [ ]:
# 清理
! kubectl delete sa readonly-user --wait=false 2>/dev/null
! kubectl delete role pod-reader --wait=false 2>/dev/null
! kubectl delete rolebinding read-pods-binding --wait=false 2>/dev/null

---

## 🎯 第3周总结

| 概念 | 一句话 |
|------|--------|
| **调度** | Filter → Score → Bind，用 affinity/taint 控制 |
| **资源管理** | requests 保底 + limits 封顶，决定 QoS |
| **健康检查** | liveness（重启）≠ readiness（摘除）≠ startup（慢启动） |
| **HPA** | 指标超阈值自动扩缩容，需 metrics-server |
| **Helm** | K8s 的 apt/yum，模板化 + values 值管理 |
| **RBAC** | SA（身份）+ Role（权限）+ Binding（绑定） |

---

## 🧪 综合练习

对之前的应用进行全面加固：

1. 所有服务配置 requests/limits
2. Web 服务配置 HPA（min 2, max 10, CPU 70%）
3. 添加 liveness + readiness 探针
4. 用 Helm 打包整个应用
5. RBAC 限制各服务的 ServiceAccount 权限
6. 验证：压测触发自动伸缩

In [ ]:
# 你的练习
pass